In [1]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(os.getenv('MONGODB_URI'))

# Lister toutes les bases
db_list = client.list_database_names()

for db_name in db_list:
    print(f"\n📌 Base : {db_name}")
    db = client[db_name]

    # Lister les collections de cette base
    collections = db.list_collection_names()

    if not collections:
        print("  (aucune collection)")
        continue

    for col in collections:
        count = db[col].count_documents({})
        print(f"  - {col} : {count} documents")

db = client['flight_delay_history_db']

#cursor = db['aviationstack_historical_landed_flights'].find().limit(100)
#df_aviation = pd.DataFrame(list(cursor))
#df_aviation.head()




📌 Base : flight_delay_db
  - aviationstack_flights : 1294 documents
  - afklm_flights : 1000 documents
  - weather_data : 70 documents

📌 Base : flight_delay_history_db
  - aviationstack_historical_landed_flights : 26 documents

📌 Base : flight_delays_test
  - test_collection : 22 documents

📌 Base : admin
  (aucune collection)

📌 Base : local
  - oplog.rs : 26 documents


In [ ]:
# Copier les bons vols
# A UTILISER UNE FOIS
# Source et destination
source_db = client['flight_delay_db']
target_db = client['flight_delay_history_db']

source_collection = source_db['aviationstack_flights']
target_collection = target_db['aviationstack_historical_landed_flights']

# Filtre : vols atterris avec arrival.actual non null
query = {
    "flight_status": "landed",
    "arrival.actual": {"$nin": [None, ""]}
}

# Récupération des documents filtrés
filtered_docs = list(source_collection.find(query))

# Insertion dans la collection cible
if filtered_docs:
    result = target_collection.insert_many(filtered_docs)
    print(f"{len(result.inserted_ids)} documents copiés.")
else:
    print("Aucun document à copier.")


24 documents copiés.


In [2]:
from pprint import pprint

db = client['flight_delay_history_db']

collection = db['aviationstack_historical_landed_flights']

# condition
query = {
    "flight_status": "landed",
    "arrival.actual": {"$ne": None}
}

# Compter les documents
count = collection.count_documents(query)
print("Nombre de documents filtrés :", count)

# Afficher les 1 derniers
for doc in collection.find(query).sort("collected_at", -1).limit(1):
    pprint(doc)


Nombre de documents filtrés : 26
{'_id': 'U24835_2026-02-01',
 'aircraft': {'iata': None,
              'icao': None,
              'icao24': '440037',
              'registration': None},
 'airline': {'iata': 'U2', 'icao': 'EZY', 'name': 'easyJet'},
 'arrival': {'actual': '2026-02-01T15:46:00+00:00',
             'actual_runway': '2026-02-01T15:46:00+00:00',
             'airport': 'Linate',
             'baggage': None,
             'delay': None,
             'estimated': '2026-02-01T15:54:00+00:00',
             'estimated_runway': '2026-02-01T15:46:00+00:00',
             'gate': None,
             'iata': 'LIN',
             'icao': 'LIML',
             'scheduled': '2026-02-01T15:50:00+00:00',
             'terminal': None,
             'timezone': 'Europe/Rome'},
 'collected_at': datetime.datetime(2026, 2, 1, 16, 38, 3, 895000),
 'departure': {'actual': '2026-02-01T14:40:00+00:00',
               'actual_runway': '2026-02-01T14:40:00+00:00',
               'airport': 'Orly',
  

In [7]:
# test ML
import os
from dotenv import load_dotenv
from pymongo import MongoClient
import pandas as pd
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))
db = client["flight_delay_history_db"]
collection = db["aviationstack_historical_landed_flights"]

# Filtre : vols atterris avec données complètes
query = {
    "flight_status": "landed",
    "arrival.actual": {"$nin": [None, ""]},
    "arrival.scheduled": {"$nin": [None, ""]},
    "departure.scheduled": {"$nin": [None, ""]},
}

docs = list(collection.find(query))
df = pd.DataFrame(docs)

# -----------------------------
# Feature engineering
# -----------------------------

def to_dt(x):
    try:
        return pd.to_datetime(x)
    except:
        return None

# Délais départ
df["dep_scheduled"] = df["departure"].apply(lambda x: to_dt(x.get("scheduled")))
df["dep_actual"] = df["departure"].apply(lambda x: to_dt(x.get("actual")))
df["dep_estimated"] = df["departure"].apply(lambda x: to_dt(x.get("estimated")))

df["departure_delay_actual"] = (df["dep_actual"] - df["dep_scheduled"]).dt.total_seconds() / 60
df["departure_delay_estimated"] = (df["dep_estimated"] - df["dep_scheduled"]).dt.total_seconds() / 60
df["departure_delay_actual"] = df["departure_delay_actual"].apply(lambda x: max(x, 0))
df["departure_delay_estimated"] = df["departure_delay_estimated"].apply(lambda x: max(x, 0))


# Délais arrivée
df["arr_scheduled"] = df["arrival"].apply(lambda x: to_dt(x.get("scheduled")))
df["arr_actual"] = df["arrival"].apply(lambda x: to_dt(x.get("actual")))
df["arr_estimated"] = df["arrival"].apply(lambda x: to_dt(x.get("estimated")))

df["arrival_delay_actual"] = (df["arr_actual"] - df["arr_scheduled"]).dt.total_seconds() / 60
df["arrival_delay_actual"] = df["arrival_delay_actual"].apply(lambda x: max(x, 0))

df["arrival_delay_estimated"] = (df["arr_estimated"] - df["arr_scheduled"]).dt.total_seconds() / 60
df["arrival_delay_estimated"] = df["arrival_delay_estimated"].apply(lambda x: max(x, 0))

# Cible
df = df.dropna(subset=["arrival_delay_actual"])
y = df["arrival_delay_actual"]

# Features temporelles
df["flight_date"] = pd.to_datetime(df["flight_date"])
df["day_of_week"] = df["flight_date"].dt.weekday
df["month"] = df["flight_date"].dt.month
df["hour_dep"] = df["dep_scheduled"].dt.hour

# Features catégorielles
df["airline_iata"] = df["airline"].apply(lambda x: x.get("iata"))
df["dep_iata"] = df["departure"].apply(lambda x: x.get("iata"))
df["arr_iata"] = df["arrival"].apply(lambda x: x.get("iata"))
df["aircraft_icao24"] = df["aircraft"].apply(lambda x: x.get("icao24"))
df["flight_number"] = df["flight"].apply(lambda x: x.get("number"))

# Dataset final
features = df[
    [
        "departure_delay_actual",
        "departure_delay_estimated",
        "arrival_delay_estimated",
        "day_of_week",
        "month",
        "hour_dep",
        "airline_iata",
        "dep_iata",
        "arr_iata",
        "aircraft_icao24",
        "flight_number",
    ]
]

# Encodage
X = pd.get_dummies(features)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Modèle
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

# Prédictions
y_pred = model.predict(X_test)

# Évaluation
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("📊 Évaluation du modèle :")
print(f"MAE  : {mae:.2f} minutes")
print(f"RMSE : {rmse:.2f} minutes")
print(f"R²   : {r2:.2f}")


📊 Évaluation du modèle :
MAE  : 0.58 minutes
RMSE : 1.39 minutes
R²   : 0.13
